# TAIL-FLY Phase A3 — numerical audit

This train-only audit reruns the locked ImageNet-R Phase A grid with float64 classifier solves, implementation-bound unit caches, per-method residuals, and an independently selected plain-TSVD control. It never opens ImageNet-R test features. Run all cells in order.

In [ ]:
# === Edit repository/Drive paths only. Do not edit protocol values. ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'feature/tail-fly-a3'
WORK_DIR = '/content/SOHO-CL'
DRIVE_ROOT = '/content/drive/MyDrive/T-SOHO'
DRIVE_TRAIN_CACHE = f'{DRIVE_ROOT}/imagenetr_train_feature_cache_seed2025'
TRAIN_CACHE_DIR = '/content/imagenetr_train_feature_cache_seed2025'
DRIVE_WTA_CACHE = f'{DRIVE_ROOT}/tail_fly_imagenetr_wta_cache_seed2025'
WTA_CACHE_DIR = '/content/tail_fly_imagenetr_wta_cache_seed2025'
OUTPUT_DIR = f'{DRIVE_ROOT}/tail_fly_imagenetr_phasea3_seed2025'
SEED = 2025
CONFIG_SHA256 = '24f2d82b5e5662bbc315bbff46cbb962f92ea7bdc41ad9cb7e2ff6d77c129b96'

In [ ]:
# Clone the exact branch and verify the locked A3 config.
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
os.chdir('/content')
repo_path = Path(WORK_DIR)
if repo_path.exists(): shutil.rmtree(repo_path)
clone = subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_GIT_URL, WORK_DIR], text=True, capture_output=True)
print(clone.stdout, clone.stderr, sep='')
assert clone.returncode == 0, 'Clone failed: confirm the A3 branch was pushed.'
os.chdir(WORK_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle.txt'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
config_path = Path('configs/tail_fly_imagenetr_train_only_a3.json')
assert hashlib.sha256(config_path.read_bytes()).hexdigest() == CONFIG_SHA256, 'Locked A3 config mismatch.'
print('repo commit:', commit)
print('GPU:', torch.cuda.get_device_name(0))
print('seed:', SEED, '| config SHA-256:', CONFIG_SHA256)

In [ ]:
# Synthetic correctness gate, including mixed-precision and cache-identity tests.
tests = ['tests/test_tail_fly_math.py', 'tests/test_tail_fly_learner.py', 'tests/test_tail_fly_phasea.py']
subprocess.run([sys.executable, '-m', 'pytest', '-q', *tests], check=True)
print('TAIL-FLY A3 correctness gate: PASS')

In [ ]:
# Restore the exact train-only feature and WTA caches used by Phase A.
def restore_cache(source_name, destination_name, required_files):
    source, destination = Path(source_name), Path(destination_name)
    assert all((source/name).is_file() for name in required_files), f'Missing Phase A Drive cache: {source}'
    if destination.exists(): shutil.rmtree(destination)
    destination.mkdir(parents=True)
    for item in sorted(source.iterdir()):
        print('COPY', item.name, f'{item.stat().st_size/2**20:.1f} MiB', flush=True)
        shutil.copy2(item, destination/item.name)
    return destination
train_cache = restore_cache(DRIVE_TRAIN_CACHE, TRAIN_CACHE_DIR, ['metadata.json', 'train.pt'])
wta_cache = restore_cache(DRIVE_WTA_CACHE, WTA_CACHE_DIR, ['metadata.json', 'projection.pt', 'train_codes.pt'])
metadata = json.loads((train_cache/'metadata.json').read_text())
assert metadata['dataset'] == 'ImageNet-R' and metadata['feature_dim'] == 768 and metadata['finite'] is True
assert metadata['test_features_materialized'] is False and not (train_cache/'test.pt').exists()
print('CACHE GATE PASS:', metadata['train_shape'], '| test.pt absent')

In [ ]:
# Locked train-only A3 run with concise live progress.
output_path = Path(OUTPUT_DIR)
output_path.mkdir(parents=True, exist_ok=True)
shutil.copy2(config_path, output_path/'locked_config.json')
command = [sys.executable, '-u', 'tools/tail_fly_phasea.py', '--config', str(config_path), '--feature-cache-dir', TRAIN_CACHE_DIR, '--code-cache-dir', WTA_CACHE_DIR, '--output-dir', OUTPUT_DIR, '--device', 'cuda', '--require-test-hidden']
print('Starting A3: exact/raw controls, then rank 64/128/256.', flush=True)
print('TASK lines show effective rank, tail fraction, mixed-precision residual, and elapsed time.', flush=True)
started = time.time()
completed = subprocess.run(command)
print(f'A3 elapsed: {(time.time()-started)/60:.1f} minutes', flush=True)
assert completed.returncode == 0, 'A3 failed; send the complete traceback without changing config.'
assert (output_path/'phasea_results.json').is_file() and not (train_cache/'test.pt').exists()
print('TAIL-FLY A3 process: COMPLETE')

In [ ]:
# Inspect the decisive comparisons and download the train-only evidence bundle.
import pandas as pd
result = json.loads((output_path/'phasea_results.json').read_text())
columns = ['method','rank','ridge_lambda','validation_average_accuracy','maximum_solver_relative_residual','persistent_state_bytes']
display(pd.DataFrame([{key: row.get(key) for key in columns} for row in result['candidates']]).sort_values(['method','validation_average_accuracy'], ascending=[True,False]))
print('selected TAIL:', json.dumps(result['selected'], indent=2))
print('best independent plain TSVD:', json.dumps(result['selected_plain_tsvd'], indent=2))
print('decision:', result['decision'])
print('gates:', json.dumps(result['gates'], indent=2))
print('diagnostics:', json.dumps(result['gate_diagnostics'], indent=2))
archive = shutil.make_archive('/content/tail_fly_imagenetr_phasea3_train_only', 'zip', root_dir=OUTPUT_DIR)
print('artifact SHA-256:', hashlib.sha256(Path(archive).read_bytes()).hexdigest())
from google.colab import files
files.download(archive)
print('STOP. Send the ZIP for audit; do not evaluate ImageNet-R test.')